In [1]:
import scvi
import scanpy as sc
import numpy as np

ImportError: cannot import name 'get_num_classes' from 'torchmetrics.utilities.data' (/Users/ingridkjaerulffjensen/envs/mlenv/lib/python3.8/site-packages/torchmetrics/utilities/data.py)

In [ ]:
## Load the data / adata
# !!! and use the adata.X matrix those should be the raw counts (?)

Setup

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    batch_key="Participant"    # corrects for patient-level variation
)

Build and train:
This will take a few minutes. You'll see a progress bar with the loss decreasing.

In [ ]:
model = scvi.model.SCVI(
    adata,
    n_latent=20,
    n_layers=2,
    n_hidden=128
)

model.train(
    max_epochs=400,
    early_stopping=True
)

Step 4 — Extract and Store

In [ ]:
adata.obsm["X_scVI"] = model.get_latent_representation()

# Recompute neighbors and UMAP from scVI space
sc.pp.neighbors(adata, use_rep="X_scVI")
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)

Step 5 — Quick Sanity Check Plot

In [ ]:
sc.pl.umap(adata, color=[
    "Coarse_Annotation",     # do cell types cluster cleanly?
    "Participant",           # should be mixed — batch correction working
    "Variant_Group",         # biological signal
    "WHO_Score_at_Peak"      # disease severity
])